<a href="https://colab.research.google.com/github/eeeewyz/Audio-course/blob/main/8_Seq2Seq_Whisper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#加载数据集
from datasets import load_dataset

dataset = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy", "clean", split="validation"
)
dataset

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

clean/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.19MB            

clean/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/73 [00:00<?, ? examples/s]

Dataset({
    features: ['file', 'audio', 'text', 'speaker_id', 'chapter_id', 'id'],
    num_rows: 73
})

In [3]:
#用whisper的base模型
import torch
from transformers import pipeline

device = "cuda:0" if torch.cuda.is_available() else "cpu"
pipe = pipeline(
    "automatic-speech-recognition", model="openai/whisper-base", device=device
)

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  290MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.81k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

In [5]:
sample=dataset[0]
#进行一次transcript
pipe(sample["audio"], max_new_tokens=256)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass

{'text': ' Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel.'}

In [6]:
#看下原本的target
sample['text']

'MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL'

In [ ]:
# 你的模型输出：

# Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel.

# Target：

# MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL

# 核心内容逐词看几乎完全一致：

# Mr. ↔ MISTER：语义相同，但字符串不同
# 模型输出有正常的大小写：Mr. Quilter ...
# 模型输出有逗号和句号：, .
# target 全部大写，而且没有标点
# 其他主要单词内容基本一致

In [8]:
#加载一个多语言数据集： Multilingual LibriSpeech (MLS) dataset
dataset = load_dataset(
    "facebook/multilingual_librispeech", "spanish", split="test", streaming=True
)
sample = next(iter(dataset))

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

In [11]:
from IPython.display import Audio
#这是个西班牙语的数据
print(sample["transcript"])
Audio(sample["audio"]["array"], rate=sample["audio"]["sampling_rate"])

y las almas buscando algún alivio se revuelven ansiosas y hacen el mundo que así resulta ser del dolor obra el dolor o la nada quien tenga corazón venga y escoja


In [12]:
sample

{'audio': <datasets.features._torchcodec.AudioDecoder at 0x7b2f38716990>,
 'original_path': 'http://www.archive.org/download/poesias_1805_librivox/poesias_038_unamuno_64kb.mp3',
 'begin_time': 427.24,
 'end_time': 443.51,
 'transcript': 'y las almas buscando algún alivio se revuelven ansiosas y hacen el mundo que así resulta ser del dolor obra el dolor o la nada quien tenga corazón venga y escoja',
 'audio_duration': 16.269999999999982,
 'speaker_id': '11266',
 'chapter_id': '10604',
 'file': '11266_10604_000000.opus',
 'id': '11266_10604_000000'}

In [14]:
#transcript下：task": "transcribe
pipe(sample["audio"], max_new_tokens=256, generate_kwargs={"task": "transcribe"})

[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'text': ' Y las almas, buscando alguna livión, se revuelven ansiosas y hacen el mundo, que así resulta ser del dolor obra. El dolor o la nada, que entenga corazón venga y escoja.'}

In [17]:
#whisper支持多任务，用下翻译的任务类型
pipe(sample["audio"], max_new_tokens=256, generate_kwargs={"task": "translate"})

[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'text': ' And the souls, looking for some relief, are relinquishing themselves in the world, which thus results in the pain of the work. The pain or nothing, who has heart, come and take it.'}